This code sends requests to Google Maps' API for drivetimes between two different locations. The returned data was used as an input for a patient choice model within a hospital market.

### Code Setup - Import Needed Libraries and Set Local Variables if Needed

In [ ]:
#Python 3.14.5

import requests #2.34.2
import pandas as pd #3.0.3
import numpy as np #2.4.6

#INSERT API KEY HERE
key = "INSERT API KEY HERE"
path = r"INSERT PATH TO CASE FOLDER"

________________

### Define Needed Functions

In [ ]:
def find_drivetime(location, address):
    distance, duration = None, None

    url = "https://maps.googleapis.com/maps/api/distancematrix/json?units=imperial&origins="+location+"&destinations="+address+"&units=imperial&departure_time=1711980000&key="+key
    payload={}
    headers = {}
    response = requests.request("GET", url, headers=headers, data=payload)
    if response.status_code not in range(200,299):
        return None, None

    try:
        results = response.json()
        distance = results['rows'][0]['elements'][0]['distance']['text']
        duration = results['rows'][0]['elements'][0]['duration_in_traffic']['value']

    except:
        pass

    return distance, duration

def add_drivetime_dataframe(row):
    row_num = str(row.name + 1)
    print(f"\rProgress: Row # "+row_num,end="")
    GPS_location = row['location']
    Hosp_Address = row['hospital_address']
    distance, duration = find_drivetime(GPS_location, Hosp_Address)
    row['distance'] = distance
    row['duration_seconds'] = duration
    return row

Use population-weighted centroids and list of hospital-zip code pairs to request drive times

In [ ]:
#Import population-weighted zip code centroid latitudes and longitudes
zips = pd.read_csv(rf'{path}\Zip_Code_Population_Weighted_Centroids.csv')
zips['STD_ZIP5'] = zips['STD_ZIP5'].astype(int)

#Import desired hospital - zip code pairs 
Pairs = pd.read_stata(rf'{path}\location pairs for drive times filtered.dta')
Pairs['patient_zip'] = Pairs['patient_zip'].astype(int)

#Merge Together
Pairs = Pairs.merge(zips, left_on='patient_zip', right_on='STD_ZIP5')
Pairs['location'] = Pairs['LATITUDE'].astype(str) + '%2C' + Pairs['LONGITUDE'].astype(str)
Pairs = Pairs[['patient_zip','hospital_id','hospital_address','adr_label','location']]

#Split into chunks
number_of_chunks = 120
df_split = np.array_split(Pairs, number_of_chunks)
del Pairs

#Request Drive Times and Save for each chunk
for i in range(0,number_of_chunks):
    #Request Drive Times
    df_split[i] = df_split[i].apply(add_drivetime_dataframe, axis=1)

    #destring distance
    df_split[i]['drive_miles'] = df_split[i]['distance'].str.extract(r'(\d+)').astype(int)

    #Save with all fields to intermediate folder
    chunk_num = i + 1
    df_split[i].to_csv(rf'{path}\Intermediate\Chunks from Minutes Version\CSV\Chunk {chunk_num}.csv', index = False)
    df_split[i].to_stata(rf'{path}\Intermediate\Chunks from Minutes Version\DTA\Chunk {chunk_num}.dta', write_index = False)

    #Keep columns of interest for Analysis Data
    df_split[i] = df_split[i][['patient_zip','hospital_id','hospital_address','adr_label','duration_seconds','drive_miles']]

    #Update user 
    print(" | Finished "+str(i+1)+" of "+str(number_of_chunks)+" chunks")

#Append chunks and save as analysis version
df = pd.read_csv(rf'{path}\Intermediate\Chunks from Minutes Version\CSV\Chunk 1.csv')
for i in range (2,number_of_chunks+1):
    df_add = pd.read_csv(rf'{path}\Intermediate\Chunks from Minutes Version\CSV\Chunk {i}.csv')
    df = pd.concat([df,df_add],ignore_index=True) 

#Save as Analysis Version
df.to_csv(rf'{path}\Intermediate\Chunks from Minutes Version\CSV\Drive Times.csv', index = False)
df.to_stata(rf'{path}\Intermediate\Chunks from Minutes Version\DTA\Drive Times.dta', write_index = False)